# VeloceReduction — one observing night

**Data flow**

`DetectorFrame → OrderGeometry → OrderMatrix → ExtractionResult`

For `extraction_mode="fibre"`, `FibreGeometry` is an additional Flat-derived input to the final extraction.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io import fits
from astropy.table import Table, vstack

from velocereduction import __version__, ReductionConfig
from velocereduction import (
    observations, detector, orders, fibres, flat, extraction,
    calibration, thorium, simlc, wavelength, diagnostics, constants
)
from velocereduction.config import prepare_reduction, setup_logging

night = "001122"
# night = "260703"
config = ReductionConfig(
    night=night,
    extraction_mode="fibre",   # "summed" or "fibre"
    diagnostics="full",
    log_level="DEBUG",
    overwrite=False,
)
paths = prepare_reduction(config, __version__)
logger = setup_logging(config, paths)
print(paths.root)

## 1. Identify observations

The observing log and FITS headers are reconciled first. This stage only decides what data are available and which CCDs should be used.

In [ ]:
reduction_input = observations.identify_observations(config, paths)
display(reduction_input)
print(f"{len(reduction_input)} observations selected for {night}")

## 2. Detector registration

Detector shifts are measured in `detector.py` relative to the reference night. The compact result is written as `detector_shifts_YYMMDD.fits`.


In [ ]:
detector_shifts = detector.measure_detector_shifts(reduction_input, config, paths)
display(detector_shifts)
fits.info(paths.detector_shifts)

## 3. Combine Flats in memory

The individual Flat `DetectorFrame`s are normalized and combined. The full 4112×4096 combined Flat is intentionally **not** saved; it is only an intermediate used to determine geometry and compact 1-D response products.


In [ ]:
combined_flats = flat.combine_flat_frames(reduction_input, config)
for ccd, frame in combined_flats.items():
    print(f"CCD{ccd}: image={frame.image.shape}, finite={np.mean(np.isfinite(frame.image)):.3%}")

## 4. Determine `OrderGeometry`

The reference-night geometry supplies the starting locations. The current Flat and measured detector shifts refine the trace and named cross-dispersion regions. The persistent product has one table row per physical echelle order.


In [ ]:
order_geometry = orders.determine_order_geometry(
    reduction_input, combined_flats, detector_shifts, config, paths
)
order_table = orders.order_geometry_table(order_geometry)
display(order_table[:10])
fits.info(paths.order_geometry)
print(paths.order_geometry)

## 5. Optional compact `FibreGeometry`

For fibre extraction only, the Flat order matrices are fitted at sparse dispersion locations. The saved model contains polynomial coefficients for bundle offset, fibre separation and common Gaussian width, plus one constant offset for each science/sky fibre. The evaluated 4112-row centres are never written to disk.


In [ ]:
flat_order_matrices = flat.extract_flat_order_matrices(combined_flats, order_geometry)
fibre_geometry = {}

if config.extraction_mode == "fibre":
    reference_file = paths.reference_product("fibre_geometry", config.reference_night)
    reference_geometry = fibres.load_fibre_geometry(reference_file) if reference_file.exists() else {}

    if paths.fibre_geometry.exists() and not config.overwrite:
        fibre_geometry = fibres.load_fibre_geometry(paths.fibre_geometry)
    else:
        fibre_geometry = fibres.fit_fibre_geometries(
            flat_order_matrices, config, reference_geometries=reference_geometry
        )
        fibres.save_fibre_geometry(paths.fibre_geometry, fibre_geometry, config)
        fibres.save_fibre_diagnostics(fibre_geometry, flat_order_matrices, config, paths)

    summary = fibres.summarise_fibre_geometry(fibre_geometry)
    display(summary)
    fits.info(paths.fibre_geometry)
    with fits.open(paths.fibre_geometry) as hdul:
        display(Table(hdul["ORDER_MODEL"].data)[:8])
        display(Table(hdul["FIBRE_OFFSETS"].data)[:30])

## 6. Summed and fibre Flat calibrations

For the summed path, the science aperture gives one 4112-pixel Flat spectrum per order; a broad Gaussian-smoothed version is the large-scale illumination/blaze-like shape and their ratio is the small-scale response.

In fibre mode, each extracted fibre is treated independently. `response_fibres` also carries relative fibre-throughput information with respect to the median science-fibre smooth Flat. No 4112×81 multiplicative response map is applied to science/calibration order matrices.


In [ ]:
flat_calibrations = flat.build_flat_calibrations(
    flat_order_matrices, fibre_geometry, config, paths
)

for filename in (
    paths.flat_summed,
    paths.flat_smooth_summed,
    paths.response_summed,
):
    print(filename.name)
    fits.info(filename)

if config.extraction_mode == "fibre":
    for filename in (
        paths.flat_fibres,
        paths.flat_smooth_fibres,
        paths.response_fibres,
    ):
        print(filename.name)
        fits.info(filename)


### Checkpoint: direct summed Flat versus recombined fibres

This is an important independent QA test. The direct summed extraction is retained as the minimally model-dependent reference; wavelength-dependent structure in the fibre/summed ratio can reveal imperfect fibre geometry or deblending.


In [ ]:
if config.extraction_mode == "fibre":
    name = next(name for name in flat_calibrations if name.startswith("ccd_2_"))
    product = flat_calibrations[name]
    geometry = fibre_geometry[name]
    science_components = {str(f) for f in constants.SCIENCE_FIBRES}

    science_idx = np.array([i for i, component in enumerate(geometry.components) if str(component) in science_components], dtype=int)
    recombined = np.nansum(product.fibre_flat[:, science_idx], axis=1)
    scale = np.nanmedian(product.summed_flat / recombined)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(product.summed_flat, label="direct summed Flat")
    ax.plot(recombined * scale, label="recombined science fibres")
    ax.set(title=name, xlabel="Dispersion pixel", ylabel="Flat counts")
    ax.set_ylim(-1,3)
    ax.legend()
    plt.show()


## 7. Extract calibration spectra in detector coordinates

The same fixed `OrderGeometry` is used for SimTh, SimLC and summed FibTh extraction. Fibre mode additionally extracts the 19 science-fibre FibTh spectra using the fixed `FibreGeometry`. These spectra are deliberately left in detector-pixel coordinates for the next wavelength-calibration stage.


In [ ]:
calibration_exposures = extraction.extract_calibration_exposures(
    reduction_input, order_geometry, fibre_geometry, config
)
calibration_files = extraction.save_calibration_exposures(
    calibration_exposures, paths.calibrations, config.night, overwrite=True
)
for filename in calibration_files:
    print(filename.relative_to(paths.root))


## 8. Wavelength calibration

The wavelength stage starts from the detector-coordinate calibration extractions above. Peak measurement and reference-line identification are source-specific (`thorium.py` and `simlc.py`); `wavelength.py` only receives identified line tables and fits the wavelength model.

The current implementation below completes the line-measurement/LSF stage and the FibTh-based static solution. The hybrid FibTh+SimLC reference solution and the fibre/time-dependent corrections are the next wavelength-calibration steps.

### 8.1 Reference inputs and bootstrap wavelength solution

The bootstrap solution is used only to identify which laboratory/comb line corresponds to a measured peak. It is not the final nightly wavelength solution. Once the reference night has been validated, `wavelength_static_001122_ccd*.fits` should become the normal bootstrap for subsequent nights.

In [ ]:
Y_BOUNDS = (0.0, 4111.0)
ORDER_BOUNDS = {
    "1": (138, 167),
    "2": (103, 140),
    "3": (65, 104),
}

# Original Murphy atlas + the editable Veloce-specific selection.
murphy_atlas_file = paths.reference_data / "thar_UVES_MM090311.dat"
veloce_atlas_file = paths.reference_data / "veloce_thorium_reference.fits"

if veloce_atlas_file.exists():
    thorium_atlas = thorium.read_veloce_thorium_atlas(veloce_atlas_file)
    print(f"Using curated Veloce Th atlas: {veloce_atlas_file.name}")
else:
    thorium_atlas = thorium.load_murphy_thorium_atlas(murphy_atlas_file)
    print("WARNING: curated Veloce Th atlas not found; using all Murphy Th lines.")

bootstrap_solution = {}
for ccd in ("1", "2", "3"):
    final_reference = (
        paths.reference_data
        / f"wavelength_static_{config.reference_night}_ccd{ccd}.fits"
    )
    legacy_bootstrap = (
        paths.reference_data
        / f"wavelength_bootstrap_{config.reference_night}_ccd{ccd}.fits"
    )

    filename = final_reference if final_reference.exists() else legacy_bootstrap
    if not filename.exists():
        raise FileNotFoundError(
            f"No bootstrap wavelength solution for CCD{ccd}. Expected either "
            f"{final_reference.name} or {legacy_bootstrap.name} in "
            f"{paths.reference_data}."
        )

    bootstrap_solution[ccd], _ = wavelength.read_wavelength_solution_fits(filename)
    print(f"CCD{ccd}: bootstrap = {filename.name}")


def detector_shift_y(ccd):
    return detector.detector_shift(detector_shifts, ccd)[1]


def summed_arrays(exposure):
    # extraction stores flux as (4112, n_orders); line fitting expects
    # (n_orders, 4112).
    return exposure.summed.flux.T, exposure.summed.variance.T

### 8.2 Calibration-line measurements

#### 8.2.1 FibTh and SimTh

Both thorium sources use the same line measurement/identification machinery. The returned table contains order, measured dispersion position and uncertainty, reference wavelength, quality flags, and detector-space FWHM.

In [ ]:
thorium_line_sets = {
    source: {ccd: [] for ccd in ("1", "2", "3")}
    for source in ("FibTh", "SimTh")
}

for source in ("FibTh", "SimTh"):
    for ccd in ("1", "2", "3"):
        for exposure_index, exposure in enumerate(calibration_exposures[source][ccd]):

            print('Working on', source, ccd, exposure.run, exposure_index)

            counts, variance = summed_arrays(exposure)

            result = thorium.measure_thorium_lines(
                counts,
                exposure.orders,
                thorium_atlas,
                variance=variance,
                source=source,
                ccd=ccd,
                exposure_index=exposure_index,
                mjd_mid=exposure.mjd_mid,
                reference_wavelength_function=bootstrap_solution[ccd].wavelength,
                detector_shift_y=detector_shift_y(ccd),
                y_bounds=Y_BOUNDS,
                minimum_reference_intensity=1.5,
            )
            thorium_line_sets[source][ccd].append(result)

            filename = (
                paths.calibrations
                / f"{source.lower()}_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
            )
            result.lines.write(filename, overwrite=True)

            n_used = np.count_nonzero(result.lines["used_for_wavelength_fit"])
            print(
                f"{source} CCD{ccd} run {exposure.run}: "
                f"{n_used}/{len(result.lines)} identified lines retained"
            )

#### 8.2.2 SimLC and the line-spread function

The initial integrated-Gaussian fit is only a peak-detection/centroid seed. `simlc.py` assigns the exact comb mode, infers the shared order-dependent Moffat/empirical eLSF, and then remeasures every identified comb centroid with the adopted profile. The effective detector-space FWHM is retained for the later resolution profile.

In [ ]:
SIMLC_REPEAT_FREQUENCY_HZ = 25.00000000e9
SIMLC_OFFSET_FREQUENCY_HZ = 9.56000000000e9

simlc_line_sets = {ccd: [] for ccd in ("2", "3")}

for ccd in ("2", "3"):
    for exposure_index, exposure in enumerate(calibration_exposures["SimLC"][ccd]):
        counts, variance = summed_arrays(exposure)

        print('Working on', source, ccd, exposure.run, exposure_index)

        result = simlc.measure_simlc_lines(
            counts,
            exposure.orders,
            variance=variance,
            ccd=ccd,
            exposure_index=exposure_index,
            mjd_mid=exposure.mjd_mid,
            reference_wavelength_function=bootstrap_solution[ccd].wavelength,
            detector_shift_y=detector_shift_y(ccd),
            y_bounds=Y_BOUNDS,
            repetition_rate_hz=SIMLC_REPEAT_FREQUENCY_HZ,
            offset_frequency_hz=SIMLC_OFFSET_FREQUENCY_HZ,
        )
        simlc_line_sets[ccd].append(result)

        line_filename = (
            paths.calibrations
            / f"simlc_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
        )
        result.lines.write(line_filename, overwrite=True)

        if result.lsf is not None:
            lsf_filename = (
                paths.calibrations
                / f"simlc_lsf_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
            )
            simlc.write_simlc_lsf_fits(
                result.lsf,
                lsf_filename,
                ccd=ccd,
                mjd_mid=exposure.mjd_mid,
                source_peak_file=line_filename,
                overwrite=True,
            )

            if config.diagnostics != "none":
                simlc.plot_simlc_lsf_qa(
                    result.lsf,
                    result.lines,
                    filename=(
                        paths.figures
                        / f"simlc_lsf_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.png"
                    ),
                )

        n_used = np.count_nonzero(result.lines["used_for_wavelength_fit"])
        print(
            f"SimLC CCD{ccd} run {exposure.run}: "
            f"{n_used}/{len(result.lines)} modes retained"
        )

### 8.3 Static summed wavelength solution

For the first static solution, choose the FibTh exposure closest to the median calibration time and fit the global $m\lambda(y,m)$ surface. The Legendre degrees are selected by spatially blocked cross-validation rather than from the training residuals.

This is intentionally the **FibTh-only baseline**. The next implementation step is to transform SimLC positions onto the summed-science coordinate and replace the FibTh constraints order-by-order where reliable SimLC coverage exists on CCDs 2 and 3.

In [ ]:
all_calibration_mjds = [
    exposure.mjd_mid
    for source in calibration_exposures.values()
    for ccd_exposures in source.values()
    for exposure in ccd_exposures
    if np.isfinite(exposure.mjd_mid)
]
t0 = float(np.median(all_calibration_mjds))
print(f"Static wavelength reference time: MJD {t0:.8f}")

static_wavelength = {}
static_fitted_lines = {}
static_validation = {}
static_reference_exposure = {}

for ccd in ("1", "2", "3"):
    candidates = calibration_exposures["FibTh"][ccd]
    if not candidates:
        print(f"CCD{ccd}: no FibTh exposure available")
        continue

    exposure_index = int(
        np.argmin([abs(exposure.mjd_mid - t0) for exposure in candidates])
    )
    exposure = candidates[exposure_index]
    lines = thorium_line_sets["FibTh"][ccd][exposure_index].lines
    static_reference_exposure[ccd] = exposure

    fit, fitted_lines, validation, chosen = (
        wavelength.fit_validated_wavelength_from_peak_table(
            lines,
            y_bounds=Y_BOUNDS,
            order_bounds=ORDER_BOUNDS[ccd],
            y_degrees=range(4, 11),
            order_degrees=range(2, 9),
            n_folds=5,
            y_blocks=10,
        )
    )

    static_wavelength[ccd] = fit.solution
    static_fitted_lines[ccd] = fitted_lines
    static_validation[ccd] = validation

    validation_file = (
        paths.calibrations
        / f"wavelength_surface_cv_{config.night}_ccd{ccd}.fits"
    )
    validation.write(validation_file, overwrite=True)

    wavelength_file = (
        paths.calibrations
        / f"wavelength_static_{config.night}_ccd{ccd}.fits"
    )
    wavelength.write_wavelength_fit_fits(
        fit,
        fitted_lines,
        wavelength_file,
        calibration_type="FibTh",
        ccd=ccd,
        mjd_mid=exposure.mjd_mid,
        source_peak_file=(
            f"fibth_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
        ),
        overwrite=True,
    )

    if config.diagnostics != "none":
        wavelength.plot_wavelength_fit_diagnostics(
            fitted_lines,
            filename=(
                paths.figures
                / f"wavelength_static_{config.night}_ccd{ccd}.png"
            ),
        )
        diagnostics.plot_wavelength_surface_validation(
            validation,
            filename=(
                paths.figures
                / f"wavelength_surface_cv_{config.night}_ccd{ccd}.png"
            ),
        )

    print(
        f"CCD{ccd}: selected {int(chosen['y_degree'])}x"
        f"{int(chosen['order_degree'])} Legendre surface; "
        f"validation RMS={float(chosen['validation_rms_pixel']):.4f} pixel"
    )

### 8.4 Fibre-dependent wavelength measurements

The reference FibTh exposure can already be measured fibre-by-fibre with the same thorium code. These line tables are the input to the forthcoming `fit_fibre_corrections()` model; no separate peak-fitting code belongs in `wavelength.py`.

In [ ]:
fibth_fibre_line_sets = {ccd: {} for ccd in ("1", "2", "3")}

if config.extraction_mode == "fibre":
    for ccd, exposure in static_reference_exposure.items():
        if exposure.fibre_flux is None:
            continue

        for fibre_index, fibre_name in enumerate(extraction.SCIENCE_FIBRES):
            counts = exposure.fibre_flux[:, :, fibre_index].T
            variance = exposure.fibre_variance[:, :, fibre_index].T

            result = thorium.measure_thorium_lines(
                counts,
                exposure.orders,
                thorium_atlas,
                variance=variance,
                source="FibTh",
                ccd=ccd,
                exposure_index=0,
                mjd_mid=exposure.mjd_mid,
                fibre=int(fibre_name),
                reference_wavelength_function=static_wavelength[ccd].wavelength,
                detector_shift_y=0.0,
                y_bounds=Y_BOUNDS,
                minimum_reference_intensity=1.5,
            )
            fibth_fibre_line_sets[ccd][int(fibre_name)] = result

        print(
            f"CCD{ccd}: measured FibTh lines for "
            f"{len(fibth_fibre_line_sets[ccd])} science fibres"
        )

# NEXT:
# fibre_wavelength_model = wavelength.fit_fibre_corrections(
#     fibth_fibre_line_sets, static_wavelength, ...
# )

### 8.5 Temporal wavelength corrections

All SimTh and SimLC exposures have already been reduced to homogeneous line tables above. The forthcoming temporal model will compare each source against its own reference epoch so that the fixed SimTh/SimLC fibre offset is not confused with temporal drift.

In [ ]:
# NEXT:
# time_wavelength_model = wavelength.fit_time_corrections(
#     simth_line_sets=thorium_line_sets["SimTh"],
#     simlc_line_sets=simlc_line_sets,
#     static_wavelength=static_wavelength,
#     ...
# )
#
# final_wavelength = wavelength.WavelengthModel(
#     static=hybrid_static_wavelength,
#     fibre=fibre_wavelength_model,
#     time=time_wavelength_model,
# )

### 8.6 Resolution profile

The peak tables persist `fwhm_pixel` and `fwhm_uncertainty_pixel`. Once the final wavelength model is fixed, the local wavelength FWHM and resolving power can be derived from

$$
\Delta\lambda_{\rm FWHM}
=
\left|\frac{d\lambda}{dy}\right|
{\rm FWHM}_{\rm pix},
\qquad
\mathcal{R}
=
\frac{\lambda}{\Delta\lambda_{\rm FWHM}}.
$$

The smooth $\mathcal{R}(y,m)$ model should therefore be fitted after the wavelength solution rather than stored as a primary line-fit quantity.

In [ ]:
resolution_measurements = {}

for ccd, lines in static_fitted_lines.items():
    used = np.asarray(lines["used_for_wavelength_fit"], bool)
    y = np.asarray(lines["y"], float)
    order = np.asarray(lines["order"], int)
    fwhm_pixel = np.asarray(lines["fwhm_pixel"], float)

    wavelength_nm = static_wavelength[ccd].wavelength(y, order)
    dispersion_nm_per_pixel = np.abs(
        static_wavelength[ccd].dispersion(y, order)
    )
    delta_lambda_nm = dispersion_nm_per_pixel * fwhm_pixel
    resolving_power = np.divide(
        wavelength_nm,
        delta_lambda_nm,
        out=np.full_like(wavelength_nm, np.nan),
        where=np.isfinite(delta_lambda_nm) & (delta_lambda_nm > 0),
    )

    resolution_measurements[ccd] = Table(
        {
            "order": order[used],
            "y": y[used],
            "wavelength_nm": wavelength_nm[used],
            "fwhm_pixel": fwhm_pixel[used],
            "resolving_power": resolving_power[used],
        }
    )

    display(resolution_measurements[ccd][:5])

# NEXT: fit a smooth resolution profile in (y, m).

## 9. Science extraction


In [ ]:
# This needs to be updated later, once we have all the calibration products, to extract and calibrate thescience exposures

# science_exposures = science.extract_science_exposures(
#     reduction_input, nightly_tramlines, flat_products,
#     wavelength_model, config, paths
# )
# print(f"{len(science_exposures)} science CCD exposures")

# if science_exposures:
#     exposure = science_exposures[0]
#     order = exposure.orders[len(exposure.orders) // 2]
#     plt.figure(figsize=(10, 3))
#     plt.plot(order.barycentric_wavelength_nm, order.flux)
#     plt.xlabel("Barycentric wavelength / nm")
#     plt.ylabel("Flux")
#     plt.title(f"{exposure.object_name} — CCD{exposure.ccd}, order {order.order}")
#     plt.show()


## One-call equivalent


In [ ]:
# from velocereduction import pipeline
# state = pipeline.reduce_night(config, version=__version__)
